# DIP Scene Builder - full offline run (Colab / Kaggle)

**Runtime -> Change runtime type -> T4 GPU** (Kaggle: Accelerator = GPU T4, Internet = ON).

Pipeline: `data -> preprocess -> segment (classical + rembg) -> features + CLIP embeddings -> palettes -> eval + figures`.
Everything is **resumable**: if the session dies, re-run the cells and finished work is skipped.

Run order: 1) set `PER_CAT = 6` for a ~3 min smoke test, 2) set `PER_CAT = 190` and *Run all* (~15 min on GPU).

In [ ]:
# ---- settings --------------------------------------------------------------
PER_CAT  = 190          # 6 = smoke test (48 images), 190 = full run (~1520 images)
REPO_URL = "https://github.com/<YOUR-USER>/dip-scene-builder.git"   # <- put your repo here, or upload the project zip instead

In [ ]:
import os, subprocess, sys
if not os.path.exists("src"):                       # not inside the project yet -> clone it
    if "<YOUR-USER>" in REPO_URL:
        raise SystemExit("Set REPO_URL above, or upload the project folder/zip into /content and unzip it.")
    subprocess.run(["git", "clone", REPO_URL, "proj"], check=True)
    os.chdir("proj")
print(os.getcwd(), os.listdir("."))

In [ ]:
# Install deps (torch is preinstalled on Colab/Kaggle). rembg uses onnxruntime-gpu when a GPU is present.
!pip install -q -r requirements-build.txt
import onnxruntime, torch
print("onnxruntime providers:", onnxruntime.get_available_providers(), "| torch cuda:", torch.cuda.is_available())
# If CUDAExecutionProvider is missing, rembg runs on CPU: still works, but ~30-40 min for the full set.

## 1. Data (ABO listings + small images, anonymous HTTP)

In [ ]:
!python -m src.data --per-cat {PER_CAT} --out data
import pandas as pd
meta = pd.read_csv("data/meta.csv"); print(meta.category.value_counts()); meta.sample(5, random_state=0)

## 2-3. Pre-process -> segment (classical vs rembg) -> cut-outs -> HSV/HOG/LBP + CLIP embeddings
`REMBG_MODEL=u2net` is the full U2-Net.

In [ ]:
import os
os.environ["REMBG_MODEL"] = "u2net"
dev = "cuda" if torch.cuda.is_available() else "cpu"
!python -m src.build --data data --out artifacts --device {dev}

## 4. Dominant-colour palettes (k-means in LAB) for every cut-out

In [ ]:
!python -m src.palette --art artifacts

## 5. Evaluation + all figures -> `results/`

In [ ]:
!python -m src.eval --data data --art artifacts
import json; print(json.dumps(json.load(open("results/metrics.json")), indent=2))

In [ ]:
from IPython.display import Image, display
for f in sorted(os.listdir("results/figures")):
    print(f); display(Image("results/figures/" + f, width=800))

## 6. Quick scene demo with the real artifacts

In [ ]:
from src.compose import Library, compose
from src.features import load_clip
lib = Library("artifacts", load_clip(dev))
r = compose(lib, "cozy reading corner", harmony=True)
print(r["scene"], r["style"], "harmony", round(r["harmony"], 3), f'{r["seconds"]:.2f}s'); r["image"]

## 7. Package: size check + download (copy `artifacts/` and `results/` into your repo / Space)

In [ ]:
!du -sh artifacts/cutouts artifacts/*.npy artifacts/*.npz
# masks/ are only needed for figure 2 and are NOT shipped to the Space -> exclude them from the zip
!zip -rq outputs.zip artifacts results -x "artifacts/masks/*"
!ls -lh outputs.zip
try:
    from google.colab import files; files.download("outputs.zip")      # Colab
except ImportError:
    print("Kaggle: outputs.zip is in the Output panel (right side) after 'Save Version'.")